# Install Depndancy

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from scipy import stats

# ==============================================================================
# 1. FIXED CONFIGURATION & PATH REGISTRY
# ==============================================================================
BATCH_RESULTS_PATH_ARCGIS = "/content/drive/MyDrive/Iran Israel War/extracted_map_layers/crater_analysis/batch_results.json"
BATCH_RESULTS_PATH_LIVEUAMAP = "/content/drive/MyDrive/Iran Israel War/crater_analysis/batch_results.json"

ARCGIS_GT_CSV = "https://docs.google.com/spreadsheets/d/1UyUwFrQzQ63kpxE9abYUIpL5thGEjXGt0AvCspE4H-8/export?format=csv&gid=176420393"
LIVEUAMAP_GT_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"

# Dataset task matrices definition root
dataset_tasks = [
    {
        "name": "ArcGIS",
        "gt_url": ARCGIS_GT_CSV,
        "geosam_json": BATCH_RESULTS_PATH_ARCGIS,
        "base_folder": "/content/drive/MyDrive/new",
        "allowed_configs": ["Outputs_3", "Outputs_4", "Outputs_2_no_labels"] # Outputs_2 skipped for ArcGIS
    },
    {
        "name": "LiveUAMap",
        "gt_url": LIVEUAMAP_GT_CSV,
        "geosam_json": BATCH_RESULTS_PATH_LIVEUAMAP,
        "base_folder": "/content/drive/MyDrive",
        "allowed_configs": ["Outputs_3", "Outputs_4", "Outputs_2", "Outputs_2_no_labels"]
    }
]

# Experimental configuration pipeline registry mapping
config_registry = {
    "Outputs_3": "llm+Seg+Depth",
    "Outputs_4": "llm+seg",
    "Outputs_2": "llm+location label",
    "Outputs_2_no_labels": "llm+no labels"
}

# ==============================================================================
# 2. CORE UTILITY FUNCTIONS
# ==============================================================================
def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

def calculate_metrics(df, pred_col, gt_col):
    """Calculates summary metrics safely while forcing type uniformity."""
    y_pred = pd.to_numeric(df[pred_col], errors='coerce').fillna(0)
    y_true = pd.to_numeric(df[gt_col], errors='coerce').fillna(0)

    diff = y_pred - y_true
    mae  = float(diff.abs().mean())
    mse  = float((diff ** 2).mean())
    rmse = float(np.sqrt(mse))
    bias = float(diff.mean())
    return mae, mse, rmse, bias

# ==============================================================================
# 3. PIPELINE EXECUTION ENGINE
# ==============================================================================
all_results = []
raw_event_records = []  # Collector for event-level metrics to run significance tests

for task in dataset_tasks:
    print(f"\n" + "="*80)
    print(f"🚀 INITIALIZING EVALUATION FOR DATASET: {task['name'].upper()}")
    print("="*80)

    # 1. Fetch Ground Truth Target DataFrame
    print("📥 Loading ground truth...")
    try:
        gt_df = pd.read_csv(task["gt_url"]).reset_index(drop=True)
        gt_df["gt_fully"]   = pd.to_numeric(gt_df["complete_building_count"], errors="coerce").fillna(0)
        gt_df["gt_partial"] = pd.to_numeric(gt_df["partial_building_count"], errors="coerce").fillna(0)
        gt_df["gt_total"]   = pd.to_numeric(gt_df["total_count"], errors="coerce").fillna(0)
        print(f"   -> Ground truth mapped successfully: {len(gt_df)} features")
    except Exception as e:
        print(f"   ❌ Critical Error loading Ground Truth. Skipping dataset. Error: {e}")
        continue

    # 2. Load Dual GeoSAM Baselines (Google vs Esri)
    print("📥 Loading and parsing Dual GeoSAM (Google & Esri) predictions...")
    try:
        if os.path.exists(task["geosam_json"]):
            geosam_raw = pd.read_json(task["geosam_json"])

            # Parse Google Components
            g_comp = pd.to_numeric(geosam_raw.get('google_complete', 0), errors='coerce').fillna(0)
            g_part = pd.to_numeric(geosam_raw.get('google_partial', 0), errors='coerce').fillna(0)
            geosam_raw['geosam_google_fully'] = g_comp
            geosam_raw['geosam_google_partial'] = g_part
            geosam_raw['geosam_google_total'] = g_comp + g_part

            # Parse Esri Components
            e_comp = pd.to_numeric(geosam_raw.get('esri_complete', 0), errors='coerce').fillna(0)
            e_part = pd.to_numeric(geosam_raw.get('esri_partial', 0), errors='coerce').fillna(0)
            geosam_raw['geosam_esri_fully'] = e_comp
            geosam_raw['geosam_esri_partial'] = e_part
            geosam_raw['geosam_esri_total'] = e_comp + e_part

            keep_cols = [
                'index',
                'geosam_google_total', 'geosam_google_fully', 'geosam_google_partial',
                'geosam_esri_total', 'geosam_esri_fully', 'geosam_esri_partial'
            ]
            geosam_df = geosam_raw[keep_cols].rename(columns={'index': 'ID'})
            print(f"   -> GeoSAM baseline vectors successfully loaded: {len(geosam_df)} entities")
        else:
            print(f"   ⚠️ GeoSAM file absent at target location. Instantiating empty template.")
            geosam_df = pd.DataFrame(columns=['ID', 'geosam_google_total', 'geosam_esri_total'])
    except Exception as e:
        print(f"   ⚠️ Failure processing GeoSAM payload schema: {e}. Instantiating empty template.")
        geosam_df = pd.DataFrame(columns=['ID', 'geosam_google_total', 'geosam_esri_total'])

    # 3. Iterate Through Authorized Pipeline Folders / Configurations
    for config_id in task["allowed_configs"]:
        config_label = config_registry[config_id]
        target_folder_path = os.path.join(task["base_folder"], config_id)

        print(f"\n📁 Sifting Pipeline Branch -> [{config_id}] ({config_label})")

        if not os.path.exists(target_folder_path):
            print(f"   ⚠️ Folder path not found. Skipping configuration.")
            continue

        parsed_csvs = sorted(
            f for f in os.listdir(target_folder_path)
            if f.startswith("nuextract_parsed_") and f.endswith(".csv")
        )
        print(f"   📂 Found {len(parsed_csvs)} individual LLM variant model outputs")

        for csv_file in parsed_csvs:
            try:
                df_llm = pd.read_csv(os.path.join(target_folder_path, csv_file)).reset_index(drop=True)
                df_llm.rename(columns={'final_calculated_radius_m': 'max_crater_radius_m', "latitude": 'lat_dec', 'longitude': 'lon_dec'}, inplace=True)

                if 'lat_dec' in df_llm.columns:
                    df_llm = df_llm[~(df_llm.lat_dec.isna())].copy()

                # --- Strict Matrix Intersection Triage ---
                merged_llm_gt = df_llm.merge(gt_df, left_index=True, right_on='ID', how='inner')
                final_df = merged_llm_gt.merge(geosam_df, on='ID', how='inner')

                if len(final_df) == 0:
                    continue

                model_name = final_df["model_id"].iloc[0] if "model_id" in final_df.columns else csv_file
                temp = final_df["temperature"].iloc[0] if "temperature" in final_df.columns else 0.0

                # --- Collect Row-Level Statistics for Paired Significance Matrix ---
                for _, row in final_df.iterrows():
                    pred_val = pd.to_numeric(row.get("extracted_total_impacted", 0), errors='coerce')
                    gt_val = pd.to_numeric(row.get("gt_total", 0), errors='coerce')
                    raw_event_records.append({
                        "dataset": task["name"],
                        "config_id": config_id,
                        "model_id": model_name,
                        "event_id": row["ID"],
                        "mae_total": abs(pred_val - gt_val)
                    })

                # --- Calculate Summary Metrics ---
                l_mae_f, l_mse_f, l_rmse_f, l_bias_f = calculate_metrics(final_df, "extracted_fully_inside", "gt_fully")
                l_mae_p, l_mse_p, l_rmse_p, l_bias_p = calculate_metrics(final_df, "extracted_partially_inside", "gt_partial")
                l_mae_t, l_mse_t, l_rmse_t, l_bias_t = calculate_metrics(final_df, "extracted_total_impacted", "gt_total")

                g_mae_t, g_mse_t, g_rmse_t, g_bias_t = calculate_metrics(final_df, "geosam_google_total", "gt_total")
                e_mae_t, e_mse_t, e_rmse_t, e_bias_t = calculate_metrics(final_df, "geosam_esri_total", "gt_total")

                # Parse Text Metadata
                sample_cols = [c for c in final_df.columns if c.startswith("output_sample_")]
                if sample_cols:
                    all_counts = []
                    for col in sample_cols:
                        counts = final_df[col].apply(count_words)
                        all_counts.extend(counts.tolist())
                    mean_words, median_words, std_words = float(np.mean(all_counts)), float(np.median(all_counts)), float(np.std(all_counts))
                else:
                    mean_words = median_words = std_words = None

                all_results.append({
                    "dataset": task["name"],
                    "config_id": config_id,
                    "config_label": config_label,
                    "source_file": csv_file,
                    "model_id": model_name,
                    "temperature": temp,
                    "n_events": len(final_df),
                    "llm_mae_total": l_mae_t, "llm_rmse_total": l_rmse_t, "llm_bias_total": l_bias_t,
                    "geosam_google_rmse_total": g_rmse_t, "geosam_esri_rmse_total": e_rmse_t,
                    "mean_word_count": mean_words
                })

                print(f"      🔹 Model: {model_name[:25]}... | Shared N: {len(final_df)} | LLM RMSE: {l_rmse_t:.2f} | Google RMSE: {g_rmse_t:.2f}")
            except Exception as e:
                print(f"      ❌ Failure compiling output dataset segment for {csv_file}: {e}")

# ==============================================================================
# 4. LEADERBOARD PRINTING
# ==============================================================================
if all_results:
    eval_df = pd.DataFrame(all_results).sort_values(by=["dataset", "config_id", "llm_rmse_total"])
    summary_save_path = "/content/drive/MyDrive/comprehensive_cross_config_dual_geosam_evaluation.csv"
    eval_df.to_csv(summary_save_path, index=False)

    print("\n" + "="*95)
    print("🏁 PERFORMANCE SUMMARY LEADERBOARD")
    print("="*95)

    display_columns = ["config_id", "model_id", "n_events", "llm_rmse_total", "llm_mae_total", "geosam_google_rmse_total", "geosam_esri_rmse_total"]
    for dataset_group in eval_df["dataset"].unique():
        print(f"\n🏆 LEADERBOARD MATRIX FOR DATASET: {dataset_group.upper()}")
        print("-" * 120)
        print(eval_df[eval_df["dataset"] == dataset_group][display_columns].to_string(index=False))
else:
    print("\n❌ Pipeline execution error: Zero records appended.")

# ==============================================================================
# 5. STATISTICAL SIGNIFICANCE PROCESSING ENGINE (WILCOXON SIGNED-RANK)
# ==============================================================================
if raw_event_records:
    print("\n" + "="*120)
    print("📊 COMPUTING QUERY-LEVEL PAIRED SIGNIFICANCE TESTS (MEDIAN OF ALL RUNS PER SCENE)")
    print("Hypothesis: Additive visual/depth cues yield a statistically significant reduction in absolute counting error.")
    print("="*120)

    raw_df = pd.DataFrame(raw_event_records)

    # Compress 5-run stochastic variance by isolating the median per unique event tracking query
    median_events = raw_df.groupby(['dataset', 'config_id', 'model_id', 'event_id'])['mae_total'].median().reset_index()

    def run_wilcoxon(df_source, dataset_name, model_name, config_base, config_treat):
        """Filters paired subsets safely to extract one-sided signed-rank distribution p-values."""
        base_slice = df_source[(df_source['dataset'] == dataset_name) & (df_source['model_id'] == model_name) & (df_source['config_id'] == config_base)]
        treat_slice = df_source[(df_source['dataset'] == dataset_name) & (df_source['model_id'] == model_name) & (df_source['config_id'] == config_treat)]

        # Match query pairs exactly on tracking event ID markers
        pairs = pd.merge(base_slice, treat_slice, on='event_id', suffixes=('_base', '_treat'))

        if len(pairs) < 5:
            return "N/A (Insaf. Data)"
        if (pairs['mae_total_base'] == pairs['mae_total_treat']).all():
            return "Zero Variance"

        try:
            # alternative='greater' confirms if baseline error distribution is significantly higher than treatment
            _, p_val = stats.wilcoxon(pairs['mae_total_base'], pairs['mae_total_treat'], alternative='greater')
            return f"{p_val:.4f}" if p_val >= 0.0001 else "< 0.0001***"
        except Exception:
            return "Execution Error"

    sig_reports = []
    unique_combinations = raw_df[['dataset', 'model_id']].drop_duplicates()

    for _, row in unique_combinations.iterrows():
        d_name = row['dataset']
        m_name = row['model_id']

        sig_reports.append({
            "Dataset": d_name,
            "Model Variant Target": m_name[:30],
            "Only vs +GeoSAM (Outputs_2_no_labels -> Outputs_4)": run_wilcoxon(median_events, d_name, m_name, "Outputs_2_no_labels", "Outputs_4"),
            "GeoSAM vs +Depth (Outputs_4 -> Outputs_3)": run_wilcoxon(median_events, d_name, m_name, "Outputs_4", "Outputs_3"),
            "Total Gain: Baseline vs Depth (Outputs_2_no_labels -> Outputs_3)": run_wilcoxon(median_events, d_name, m_name, "Outputs_2_no_labels", "Outputs_3")
        })

    print(pd.DataFrame(sig_reports).to_markdown(index=False))
    print("\n*Note: Values < 0.05 satisfy standard confidence thresholds ($p < 0.05$). Lower values strongly validate the regularizer.")


🚀 INITIALIZING EVALUATION FOR DATASET: ARCGIS
📥 Loading ground truth...
   -> Ground truth mapped successfully: 892 features
📥 Loading and parsing Dual GeoSAM (Google & Esri) predictions...
   -> GeoSAM baseline vectors successfully loaded: 890 entities

📁 Sifting Pipeline Branch -> [Outputs_3] (llm+Seg+Depth)
   📂 Found 3 individual LLM variant model outputs
      🔹 Model: nuextract_parsed_Qwen_Qwe... | Shared N: 890 | LLM RMSE: 4.89 | Google RMSE: 4.39
      🔹 Model: nuextract_parsed_google_g... | Shared N: 890 | LLM RMSE: 3.78 | Google RMSE: 4.39
      🔹 Model: nuextract_parsed_sakamaki... | Shared N: 890 | LLM RMSE: 4.28 | Google RMSE: 4.39

📁 Sifting Pipeline Branch -> [Outputs_4] (llm+seg)
   📂 Found 3 individual LLM variant model outputs
      🔹 Model: nuextract_parsed_Qwen_Qwe... | Shared N: 890 | LLM RMSE: 5.41 | Google RMSE: 4.39
      🔹 Model: nuextract_parsed_google_g... | Shared N: 890 | LLM RMSE: 4.62 | Google RMSE: 4.39
      🔹 Model: nuextract_parsed_sakamaki... | Shar

# Stat Sig

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from scipy import stats

# ==============================================================================
# 1. FIXED CONFIGURATION & PATH REGISTRY
# ==============================================================================
BATCH_RESULTS_PATH_ARCGIS = "/content/drive/MyDrive/Iran Israel War/extracted_map_layers/crater_analysis/batch_results.json"
BATCH_RESULTS_PATH_LIVEUAMAP = "/content/drive/MyDrive/Iran Israel War/crater_analysis/batch_results.json"

ARCGIS_GT_CSV = "https://docs.google.com/spreadsheets/d/1UyUwFrQzQ63kpxE9abYUIpL5thGEjXGt0AvCspE4H-8/export?format=csv&gid=176420393"
LIVEUAMAP_GT_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"

# Dataset task matrices definition root
dataset_tasks = [
    {
        "name": "ArcGIS",
        "gt_url": ARCGIS_GT_CSV,
        "geosam_json": BATCH_RESULTS_PATH_ARCGIS,
        "base_folder": "/content/drive/MyDrive/new",
        "allowed_configs": ["Outputs_3", "Outputs_4", "Outputs_2_no_labels"] # Outputs_2 skipped for ArcGIS
    },
    {
        "name": "LiveUAMap",
        "gt_url": LIVEUAMAP_GT_CSV,
        "geosam_json": BATCH_RESULTS_PATH_LIVEUAMAP,
        "base_folder": "/content/drive/MyDrive",
        "allowed_configs": ["Outputs_3", "Outputs_4", "Outputs_2", "Outputs_2_no_labels"]
    }
]

# Experimental configuration pipeline registry mapping
config_registry = {
    "Outputs_3": "llm+Seg+Depth",
    "Outputs_4": "llm+seg",
    "Outputs_2": "llm+location label",
    "Outputs_2_no_labels": "llm+no labels"
}

# ==============================================================================
# 2. CORE UTILITY FUNCTIONS
# ==============================================================================
def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

def calculate_metrics(df, pred_col, gt_col):
    """Calculates summary metrics safely while forcing type uniformity."""
    y_pred = pd.to_numeric(df[pred_col], errors='coerce').fillna(0)
    y_true = pd.to_numeric(df[gt_col], errors='coerce').fillna(0)

    diff = y_pred - y_true
    mae  = float(diff.abs().mean())
    mse  = float((diff ** 2).mean())
    rmse = float(np.sqrt(mse))
    bias = float(diff.mean())
    return mae, mse, rmse, bias

# ==============================================================================
# 3. PIPELINE EXECUTION ENGINE
# ==============================================================================
all_results = []
raw_event_records = []  # Collector for event-level metrics to run significance tests

for task in dataset_tasks:
    print(f"\n" + "="*80)
    print(f"🚀 INITIALIZING EVALUATION FOR DATASET: {task['name'].upper()}")
    print("="*80)

    # 1. Fetch Ground Truth Target DataFrame
    print("📥 Loading ground truth...")
    try:
        gt_df = pd.read_csv(task["gt_url"]).reset_index(drop=True)
        gt_df["gt_fully"]   = pd.to_numeric(gt_df["complete_building_count"], errors="coerce").fillna(0)
        gt_df["gt_partial"] = pd.to_numeric(gt_df["partial_building_count"], errors="coerce").fillna(0)
        gt_df["gt_total"]   = pd.to_numeric(gt_df["total_count"], errors="coerce").fillna(0)
        print(f"   -> Ground truth mapped successfully: {len(gt_df)} features")
    except Exception as e:
        print(f"   ❌ Critical Error loading Ground Truth. Skipping dataset. Error: {e}")
        continue

    # 2. Load Dual GeoSAM Baselines (Google vs Esri)
    print("📥 Loading and parsing Dual GeoSAM (Google & Esri) predictions...")
    try:
        if os.path.exists(task["geosam_json"]):
            geosam_raw = pd.read_json(task["geosam_json"])

            # Parse Google Components
            g_comp = pd.to_numeric(geosam_raw.get('google_complete', 0), errors='coerce').fillna(0)
            g_part = pd.to_numeric(geosam_raw.get('google_partial', 0), errors='coerce').fillna(0)
            geosam_raw['geosam_google_fully'] = g_comp
            geosam_raw['geosam_google_partial'] = g_part
            geosam_raw['geosam_google_total'] = g_comp + g_part

            # Parse Esri Components
            e_comp = pd.to_numeric(geosam_raw.get('esri_complete', 0), errors='coerce').fillna(0)
            e_part = pd.to_numeric(geosam_raw.get('esri_partial', 0), errors='coerce').fillna(0)
            geosam_raw['geosam_esri_fully'] = e_comp
            geosam_raw['geosam_esri_partial'] = e_part
            geosam_raw['geosam_esri_total'] = e_comp + e_part

            keep_cols = [
                'index',
                'geosam_google_total', 'geosam_google_fully', 'geosam_google_partial',
                'geosam_esri_total', 'geosam_esri_fully', 'geosam_esri_partial'
            ]
            geosam_df = geosam_raw[keep_cols].rename(columns={'index': 'ID'})
            print(f"   -> GeoSAM baseline vectors successfully loaded: {len(geosam_df)} entities")
        else:
            print(f"   ⚠️ GeoSAM file absent at target location. Instantiating empty template.")
            geosam_df = pd.DataFrame(columns=['ID', 'geosam_google_total', 'geosam_esri_total'])
    except Exception as e:
        print(f"   ⚠️ Failure processing GeoSAM payload schema: {e}. Instantiating empty template.")
        geosam_df = pd.DataFrame(columns=['ID', 'geosam_google_total', 'geosam_esri_total'])

    # 3. Iterate Through Authorized Pipeline Folders / Configurations
    for config_id in task["allowed_configs"]:
        config_label = config_registry[config_id]
        target_folder_path = os.path.join(task["base_folder"], config_id)

        print(f"\n📁 Sifting Pipeline Branch -> [{config_id}] ({config_label})")

        if not os.path.exists(target_folder_path):
            print(f"   ⚠️ Folder path not found. Skipping configuration.")
            continue

        parsed_csvs = sorted(
            f for f in os.listdir(target_folder_path)
            if f.startswith("nuextract_parsed_") and f.endswith(".csv")
        )
        print(f"   📂 Found {len(parsed_csvs)} individual LLM variant model outputs")

        for csv_file in parsed_csvs:
            try:
                df_llm = pd.read_csv(os.path.join(target_folder_path, csv_file)).reset_index(drop=True)
                df_llm.rename(columns={'final_calculated_radius_m': 'max_crater_radius_m', "latitude": 'lat_dec', 'longitude': 'lon_dec'}, inplace=True)

                if 'lat_dec' in df_llm.columns:
                    df_llm = df_llm[~(df_llm.lat_dec.isna())].copy()

                # --- Strict Matrix Intersection Triage ---
                merged_llm_gt = df_llm.merge(gt_df, left_index=True, right_on='ID', how='inner')
                final_df = merged_llm_gt.merge(geosam_df, on='ID', how='inner')

                if len(final_df) == 0:
                    continue

                model_name = final_df["model_id"].iloc[0] if "model_id" in final_df.columns else csv_file
                temp = final_df["temperature"].iloc[0] if "temperature" in final_df.columns else 0.0

                # --- Collect Row-Level Statistics for Paired Significance Matrix ---
                for _, row in final_df.iterrows():
                    pred_val = pd.to_numeric(row.get("extracted_total_impacted", 0), errors='coerce')
                    gt_val = pd.to_numeric(row.get("gt_total", 0), errors='coerce')
                    raw_event_records.append({
                        "dataset": task["name"],
                        "config_id": config_id,
                        "model_id": model_name,
                        "event_id": row["ID"],
                        "pred_val": pred_val,  # <--- Store raw prediction
                        "gt_val": gt_val       # <--- Store ground truth
                    })

                # --- Calculate Summary Metrics ---
                l_mae_f, l_mse_f, l_rmse_f, l_bias_f = calculate_metrics(final_df, "extracted_fully_inside", "gt_fully")
                l_mae_p, l_mse_p, l_rmse_p, l_bias_p = calculate_metrics(final_df, "extracted_partially_inside", "gt_partial")
                l_mae_t, l_mse_t, l_rmse_t, l_bias_t = calculate_metrics(final_df, "extracted_total_impacted", "gt_total")

                g_mae_t, g_mse_t, g_rmse_t, g_bias_t = calculate_metrics(final_df, "geosam_google_total", "gt_total")
                e_mae_t, e_mse_t, e_rmse_t, e_bias_t = calculate_metrics(final_df, "geosam_esri_total", "gt_total")

                # Parse Text Metadata
                sample_cols = [c for c in final_df.columns if c.startswith("output_sample_")]
                if sample_cols:
                    all_counts = []
                    for col in sample_cols:
                        counts = final_df[col].apply(count_words)
                        all_counts.extend(counts.tolist())
                    mean_words, median_words, std_words = float(np.mean(all_counts)), float(np.median(all_counts)), float(np.std(all_counts))
                else:
                    mean_words = median_words = std_words = None

                all_results.append({
                    "dataset": task["name"],
                    "config_id": config_id,
                    "config_label": config_label,
                    "source_file": csv_file,
                    "model_id": model_name,
                    "temperature": temp,
                    "n_events": len(final_df),
                    "llm_mae_total": l_mae_t, "llm_rmse_total": l_rmse_t, "llm_bias_total": l_bias_t,
                    "geosam_google_rmse_total": g_rmse_t, "geosam_esri_rmse_total": e_rmse_t,
                    "mean_word_count": mean_words
                })

                print(f"       🔹 Model: {model_name[:25]}... | Shared N: {len(final_df)} | LLM RMSE: {l_rmse_t:.2f} | Google RMSE: {g_rmse_t:.2f}")
            except Exception as e:
                print(f"       ❌ Failure compiling output dataset segment for {csv_file}: {e}")

# ==============================================================================
# 4. LEADERBOARD PRINTING
# ==============================================================================
if all_results:
    eval_df = pd.DataFrame(all_results).sort_values(by=["dataset", "config_id", "llm_rmse_total"])
    summary_save_path = "/content/drive/MyDrive/comprehensive_cross_config_dual_geosam_evaluation.csv"
    eval_df.to_csv(summary_save_path, index=False)

    print("\n" + "="*95)
    print("🏁 PERFORMANCE SUMMARY LEADERBOARD")
    print("="*95)

    display_columns = ["config_id", "model_id", "n_events", "llm_rmse_total", "llm_mae_total", "geosam_google_rmse_total", "geosam_esri_rmse_total"]
    for dataset_group in eval_df["dataset"].unique():
        print(f"\n🏆 LEADERBOARD MATRIX FOR DATASET: {dataset_group.upper()}")
        print("-" * 120)
        print(eval_df[eval_df["dataset"] == dataset_group][display_columns].to_string(index=False))
else:
    print("\n❌ Pipeline execution error: Zero records appended.")

# ==============================================================================
# 5. STATISTICAL SIGNIFICANCE PROCESSING ENGINE (WILCOXON SIGNED-RANK)
# ==============================================================================
if raw_event_records:
    print("\n" + "="*120)
    print("📊 COMPUTING QUERY-LEVEL PAIRED SIGNIFICANCE TESTS (MEDIAN OF ALL RUNS PER SCENE)")
    print("Hypothesis: Additive visual/depth cues yield a statistically significant reduction in absolute counting error.")
    print("="*120)

    raw_df = pd.DataFrame(raw_event_records)

    # CORRECT METHOD: Compress 5-run stochastic variance by isolating the median prediction per unique event
    median_events = raw_df.groupby(['dataset', 'config_id', 'model_id', 'event_id']).agg({
        'pred_val': 'median',
        'gt_val': 'first' # Ground truth remains consistent across runs
    }).reset_index()

    # Calculate the Absolute Error of that stable median prediction
    median_events['mae_total'] = abs(median_events['pred_val'] - median_events['gt_val'])

    def run_wilcoxon(df_source, dataset_name, model_name, config_base, config_treat):
        """Filters paired subsets safely to extract one-sided signed-rank distribution p-values."""
        base_slice = df_source[(df_source['dataset'] == dataset_name) & (df_source['model_id'] == model_name) & (df_source['config_id'] == config_base)]
        treat_slice = df_source[(df_source['dataset'] == dataset_name) & (df_source['model_id'] == model_name) & (df_source['config_id'] == config_treat)]

        # Match query pairs exactly on tracking event ID markers
        pairs = pd.merge(base_slice, treat_slice, on='event_id', suffixes=('_base', '_treat'))

        if len(pairs) < 5:
            return "N/A (Insaf. Data)"
        if (pairs['mae_total_base'] == pairs['mae_total_treat']).all():
            return "Zero Variance"

        try:
            # alternative='greater' confirms if baseline error distribution is significantly higher than treatment
            _, p_val = stats.wilcoxon(pairs['mae_total_base'], pairs['mae_total_treat'], alternative='greater')
            return f"{p_val:.4f}" if p_val >= 0.0001 else "< 0.0001***"
        except Exception:
            return "Execution Error"

    sig_reports = []
    unique_combinations = raw_df[['dataset', 'model_id']].drop_duplicates()

    for _, row in unique_combinations.iterrows():
        d_name = row['dataset']
        m_name = row['model_id']

        sig_reports.append({
            "Dataset": d_name,
            "Model Variant Target": m_name[:30],
            "Only vs +GeoSAM (Outputs_2_no_labels -> Outputs_4)": run_wilcoxon(median_events, d_name, m_name, "Outputs_2_no_labels", "Outputs_4"),
            "GeoSAM vs +Depth (Outputs_4 -> Outputs_3)": run_wilcoxon(median_events, d_name, m_name, "Outputs_4", "Outputs_3"),
            "Total Gain: Baseline vs Depth (Outputs_2_no_labels -> Outputs_3)": run_wilcoxon(median_events, d_name, m_name, "Outputs_2_no_labels", "Outputs_3")
        })

    print(pd.DataFrame(sig_reports).to_markdown(index=False))
    print("\n*Note: Values < 0.05 satisfy standard confidence thresholds (p < 0.05). Lower values strongly validate the regularizer.")


🚀 INITIALIZING EVALUATION FOR DATASET: ARCGIS
📥 Loading ground truth...
   -> Ground truth mapped successfully: 892 features
📥 Loading and parsing Dual GeoSAM (Google & Esri) predictions...
   -> GeoSAM baseline vectors successfully loaded: 890 entities

📁 Sifting Pipeline Branch -> [Outputs_3] (llm+Seg+Depth)
   📂 Found 3 individual LLM variant model outputs
       🔹 Model: nuextract_parsed_Qwen_Qwe... | Shared N: 890 | LLM RMSE: 4.89 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_google_g... | Shared N: 890 | LLM RMSE: 3.78 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_sakamaki... | Shared N: 890 | LLM RMSE: 4.28 | Google RMSE: 4.39

📁 Sifting Pipeline Branch -> [Outputs_4] (llm+seg)
   📂 Found 3 individual LLM variant model outputs
       🔹 Model: nuextract_parsed_Qwen_Qwe... | Shared N: 890 | LLM RMSE: 5.41 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_google_g... | Shared N: 890 | LLM RMSE: 4.62 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_sakamaki... 

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from scipy import stats

# ==============================================================================
# 1. FIXED CONFIGURATION & PATH REGISTRY
# ==============================================================================
BATCH_RESULTS_PATH_ARCGIS = "/content/drive/MyDrive/Iran Israel War/extracted_map_layers/crater_analysis/batch_results.json"
BATCH_RESULTS_PATH_LIVEUAMAP = "/content/drive/MyDrive/Iran Israel War/crater_analysis/batch_results.json"

ARCGIS_GT_CSV = "https://docs.google.com/spreadsheets/d/1UyUwFrQzQ63kpxE9abYUIpL5thGEjXGt0AvCspE4H-8/export?format=csv&gid=176420393"
LIVEUAMAP_GT_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"

# Dataset task matrices definition root
dataset_tasks = [
    {
        "name": "ArcGIS",
        "gt_url": ARCGIS_GT_CSV,
        "geosam_json": BATCH_RESULTS_PATH_ARCGIS,
        "base_folder": "/content/drive/MyDrive/new",
        "allowed_configs": ["Outputs_3", "Outputs_4", "Outputs_2_no_labels"] # Outputs_2 skipped for ArcGIS
    },
    {
        "name": "LiveUAMap",
        "gt_url": LIVEUAMAP_GT_CSV,
        "geosam_json": BATCH_RESULTS_PATH_LIVEUAMAP,
        "base_folder": "/content/drive/MyDrive",
        "allowed_configs": ["Outputs_3", "Outputs_4", "Outputs_2", "Outputs_2_no_labels"]
    }
]

# Experimental configuration pipeline registry mapping
config_registry = {
    "Outputs_3": "llm+Seg+Depth",
    "Outputs_4": "llm+seg",
    "Outputs_2": "llm+location label",
    "Outputs_2_no_labels": "llm+no labels"
}

# ==============================================================================
# 2. CORE UTILITY FUNCTIONS
# ==============================================================================
def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

def calculate_metrics(df, pred_col, gt_col):
    """Calculates summary metrics safely while forcing type uniformity."""
    y_pred = pd.to_numeric(df[pred_col], errors='coerce').fillna(0)
    y_true = pd.to_numeric(df[gt_col], errors='coerce').fillna(0)

    diff = y_pred - y_true
    mae  = float(diff.abs().mean())
    mse  = float((diff ** 2).mean())
    rmse = float(np.sqrt(mse))
    bias = float(diff.mean())
    return mae, mse, rmse, bias

# ==============================================================================
# 3. PIPELINE EXECUTION ENGINE
# ==============================================================================
all_results = []
raw_event_records = []  # Collector for event-level metrics to run significance tests

for task in dataset_tasks:
    print(f"\n" + "="*80)
    print(f"🚀 INITIALIZING EVALUATION FOR DATASET: {task['name'].upper()}")
    print("="*80)

    # 1. Fetch Ground Truth Target DataFrame
    print("📥 Loading ground truth...")
    try:
        gt_df = pd.read_csv(task["gt_url"]).reset_index(drop=True)
        gt_df["gt_fully"]   = pd.to_numeric(gt_df["complete_building_count"], errors="coerce").fillna(0)
        gt_df["gt_partial"] = pd.to_numeric(gt_df["partial_building_count"], errors="coerce").fillna(0)
        gt_df["gt_total"]   = pd.to_numeric(gt_df["total_count"], errors="coerce").fillna(0)
        print(f"   -> Ground truth mapped successfully: {len(gt_df)} features")
    except Exception as e:
        print(f"   ❌ Critical Error loading Ground Truth. Skipping dataset. Error: {e}")
        continue

    # 2. Load Dual GeoSAM Baselines (Google vs Esri)
    print("📥 Loading and parsing Dual GeoSAM (Google & Esri) predictions...")
    try:
        if os.path.exists(task["geosam_json"]):
            geosam_raw = pd.read_json(task["geosam_json"])

            # Parse Google Components
            g_comp = pd.to_numeric(geosam_raw.get('google_complete', 0), errors='coerce').fillna(0)
            g_part = pd.to_numeric(geosam_raw.get('google_partial', 0), errors='coerce').fillna(0)
            geosam_raw['geosam_google_fully'] = g_comp
            geosam_raw['geosam_google_partial'] = g_part
            geosam_raw['geosam_google_total'] = g_comp + g_part

            # Parse Esri Components
            e_comp = pd.to_numeric(geosam_raw.get('esri_complete', 0), errors='coerce').fillna(0)
            e_part = pd.to_numeric(geosam_raw.get('esri_partial', 0), errors='coerce').fillna(0)
            geosam_raw['geosam_esri_fully'] = e_comp
            geosam_raw['geosam_esri_partial'] = e_part
            geosam_raw['geosam_esri_total'] = e_comp + e_part

            keep_cols = [
                'index',
                'geosam_google_total', 'geosam_google_fully', 'geosam_google_partial',
                'geosam_esri_total', 'geosam_esri_fully', 'geosam_esri_partial'
            ]
            geosam_df = geosam_raw[keep_cols].rename(columns={'index': 'ID'})
            print(f"   -> GeoSAM baseline vectors successfully loaded: {len(geosam_df)} entities")
        else:
            print(f"   ⚠️ GeoSAM file absent at target location. Instantiating empty template.")
            geosam_df = pd.DataFrame(columns=['ID', 'geosam_google_total', 'geosam_esri_total'])
    except Exception as e:
        print(f"   ⚠️ Failure processing GeoSAM payload schema: {e}. Instantiating empty template.")
        geosam_df = pd.DataFrame(columns=['ID', 'geosam_google_total', 'geosam_esri_total'])

    # 3. Iterate Through Authorized Pipeline Folders / Configurations
    for config_id in task["allowed_configs"]:
        config_label = config_registry[config_id]
        target_folder_path = os.path.join(task["base_folder"], config_id)

        print(f"\n📁 Sifting Pipeline Branch -> [{config_id}] ({config_label})")

        if not os.path.exists(target_folder_path):
            print(f"   ⚠️ Folder path not found. Skipping configuration.")
            continue

        parsed_csvs = sorted(
            f for f in os.listdir(target_folder_path)
            if f.startswith("nuextract_parsed_") and f.endswith(".csv")
        )
        print(f"   📂 Found {len(parsed_csvs)} individual LLM variant model outputs")

        for csv_file in parsed_csvs:
            try:
                df_llm = pd.read_csv(os.path.join(target_folder_path, csv_file)).reset_index(drop=True)
                df_llm.rename(columns={'final_calculated_radius_m': 'max_crater_radius_m', "latitude": 'lat_dec', 'longitude': 'lon_dec'}, inplace=True)

                if 'lat_dec' in df_llm.columns:
                    df_llm = df_llm[~(df_llm.lat_dec.isna())].copy()

                # --- Strict Matrix Intersection Triage ---
                merged_llm_gt = df_llm.merge(gt_df, left_index=True, right_on='ID', how='inner')
                final_df = merged_llm_gt.merge(geosam_df, on='ID', how='inner')

                if len(final_df) == 0:
                    continue

                model_name = final_df["model_id"].iloc[0] if "model_id" in final_df.columns else csv_file
                temp = final_df["temperature"].iloc[0] if "temperature" in final_df.columns else 0.0

                # --- Collect Row-Level Statistics for Paired Significance Matrix ---
                for _, row in final_df.iterrows():
                    pred_val = pd.to_numeric(row.get("extracted_total_impacted", 0), errors='coerce')
                    gt_val = pd.to_numeric(row.get("gt_total", 0), errors='coerce')
                    raw_event_records.append({
                        "dataset": task["name"],
                        "config_id": config_id,
                        "model_id": model_name,
                        "event_id": row["ID"],
                        "pred_val": pred_val,
                        "gt_val": gt_val
                    })

                # --- Calculate Summary Metrics ---
                l_mae_t, l_mse_t, l_rmse_t, l_bias_t = calculate_metrics(final_df, "extracted_total_impacted", "gt_total")
                g_mae_t, g_mse_t, g_rmse_t, g_bias_t = calculate_metrics(final_df, "geosam_google_total", "gt_total")
                e_mae_t, e_mse_t, e_rmse_t, e_bias_t = calculate_metrics(final_df, "geosam_esri_total", "gt_total")

                # Parse Text Metadata
                sample_cols = [c for c in final_df.columns if c.startswith("output_sample_")]
                if sample_cols:
                    all_counts = []
                    for col in sample_cols:
                        counts = final_df[col].apply(count_words)
                        all_counts.extend(counts.tolist())
                    mean_words, median_words, std_words = float(np.mean(all_counts)), float(np.median(all_counts)), float(np.std(all_counts))
                else:
                    mean_words = median_words = std_words = None

                all_results.append({
                    "dataset": task["name"],
                    "config_id": config_id,
                    "config_label": config_label,
                    "source_file": csv_file,
                    "model_id": model_name,
                    "temperature": temp,
                    "n_events": len(final_df),
                    "llm_mae_total": l_mae_t,
                    "llm_mse_total": l_mse_t,
                    "llm_rmse_total": l_rmse_t,
                    "geosam_google_mae_total": g_mae_t,
                    "geosam_google_mse_total": g_mse_t,
                    "geosam_esri_mae_total": e_mae_t,
                    "geosam_esri_mse_total": e_mse_t,
                    "mean_word_count": mean_words
                })

                print(f"       🔹 Model: {model_name[:25]}... | Shared N: {len(final_df)} | LLM RMSE: {l_rmse_t:.2f} | Google RMSE: {g_rmse_t:.2f}")
            except Exception as e:
                print(f"       ❌ Failure compiling output dataset segment for {csv_file}: {e}")

# ==============================================================================
# 4. LEADERBOARD PRINTING
# ==============================================================================
if all_results:
    eval_df = pd.DataFrame(all_results).sort_values(by=["dataset", "config_id", "llm_rmse_total"])
    summary_save_path = "/content/drive/MyDrive/comprehensive_cross_config_dual_geosam_evaluation.csv"
    eval_df.to_csv(summary_save_path, index=False)

    print("\n" + "="*95)
    print("🏁 PERFORMANCE SUMMARY LEADERBOARD")
    print("="*95)

    display_columns = ["config_id", "model_id", "n_events", "llm_rmse_total", "llm_mae_total", "llm_mse_total"]
    for dataset_group in eval_df["dataset"].unique():
        print(f"\n🏆 LEADERBOARD MATRIX FOR DATASET: {dataset_group.upper()}")
        print("-" * 120)
        print(eval_df[eval_df["dataset"] == dataset_group][display_columns].to_string(index=False))
else:
    print("\n❌ Pipeline execution error: Zero records appended.")

# ==============================================================================
# 5. STATISTICAL SIGNIFICANCE PROCESSING ENGINE (WILCOXON SIGNED-RANK)
# ==============================================================================
if raw_event_records:
    print("\n" + "="*120)
    print("📊 COMPUTING QUERY-LEVEL PAIRED SIGNIFICANCE TESTS")
    print("="*120)

    raw_df = pd.DataFrame(raw_event_records)
    median_events = raw_df.groupby(['dataset', 'config_id', 'model_id', 'event_id']).agg({
        'pred_val': 'median',
        'gt_val': 'first'
    }).reset_index()

    median_events['mae_total'] = abs(median_events['pred_val'] - median_events['gt_val'])

    def run_wilcoxon(df_source, dataset_name, model_name, config_base, config_treat):
        base_slice = df_source[(df_source['dataset'] == dataset_name) & (df_source['model_id'] == model_name) & (df_source['config_id'] == config_base)]
        treat_slice = df_source[(df_source['dataset'] == dataset_name) & (df_source['model_id'] == model_name) & (df_source['config_id'] == config_treat)]
        pairs = pd.merge(base_slice, treat_slice, on='event_id', suffixes=('_base', '_treat'))

        if len(pairs) < 5: return "N/A"
        if (pairs['mae_total_base'] == pairs['mae_total_treat']).all(): return "Zero Var"

        try:
            _, p_val = stats.wilcoxon(pairs['mae_total_base'], pairs['mae_total_treat'], alternative='greater')
            return f"{p_val:.4f}" if p_val >= 0.0001 else "< 0.0001***"
        except Exception:
            return "Error"

    sig_reports = []
    unique_combinations = raw_df[['dataset', 'model_id']].drop_duplicates()
    for _, row in unique_combinations.iterrows():
        d_name, m_name = row['dataset'], row['model_id']
        sig_reports.append({
            "Dataset": d_name, "Model": m_name[:30],
            "No Lbls vs +Seg": run_wilcoxon(median_events, d_name, m_name, "Outputs_2_no_labels", "Outputs_4"),
            "+Seg vs +Depth": run_wilcoxon(median_events, d_name, m_name, "Outputs_4", "Outputs_3")
        })

# ==============================================================================
# 6. LATEX GENERATION ENGINE
# ==============================================================================
def identify_model_meta(raw_name):
    raw = str(raw_name).lower()
    if 'claude' in raw or 'qwen dist' in raw: return "Claude (Qwen Dist)", r"\shortstack{35B \\ \scriptsize \textit{MoE}}"
    if 'qwen' in raw: return "Qwen 3.6 Baseline", r"\shortstack{35B \\ \scriptsize \textit{MoE}}"
    if 'gemma' in raw: return "Google Gemma 4", r"\shortstack{31B \\ \scriptsize \textit{Dense}}"
    if 'cosmos' in raw or 'nvidia' in raw: return "Nvidia Cosmos", r"\shortstack{32B \\ \scriptsize \textit{Dense}}"
    if 'zhipu' in raw or 'glm' in raw: return "Zhipu GLM 4.6V", r"\shortstack{10B \\ \scriptsize \textit{Dense}}"
    return str(raw_name), r"\shortstack{- \\ \scriptsize \textit{-}}"

def format_latex_val(val, vals_list, best_baseline):
    if pd.isna(val): return "-"
    sorted_vals = sorted([v for v in vals_list if not pd.isna(v)])
    is_best = (len(sorted_vals) > 0 and val == sorted_vals[0])
    is_second = (len(sorted_vals) > 1 and val == sorted_vals[1])
    beats_baseline = (val < best_baseline)

    s = f"{val:.2f}"
    if is_best: s = f"\\mathbf{{{s}}}"
    elif is_second: s = f"\\underline{{{s}}}"

    return f"${s}^{{*}}$" if beats_baseline else f"${s}$"

if all_results:
    print("\n" + "="*120)
    print("📝 GENERATING PUBLICATION-READY LATEX TABLE")
    print("="*120 + "\n")

    latex_out = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\small",
        r"\caption{\textbf{Multimodal Ablation Performance across LiveUAMap and ArcGIS Datasets.} Comprehensive evaluation of MAE and MSE metrics across nested visual feature configurations (explicit text \textit{Labels}, unassisted \textit{No Lbls}, 2D segmentation overlays \textit{+Seg}, and composite depth maps \textit{+Depth}). Note that for notation brevity in headers, the \textit{+Depth} designation denotes a composite input combining relative depth maps alongside 2D segmentation overlays rather than standalone depth. Panel~A details results on the crowdsourced LiveUAMap dataset; Panel~B details results on the high-precision ArcGIS dataset. Bold formatting indicates the best result and underlining indicates second-best within each model's configuration row or cohort. Asterisks ($^*$) denote LVLM configurations that outperform the highest-performing deterministic SAMGeo baseline on the corresponding dataset.}",
        r"\label{tab:combined_ablation_performance}",
        ""
    ]

    # Target configurations to render
    configs = [("Outputs_2", "Labels"), ("Outputs_2_no_labels", "No Lbls"), ("Outputs_4", "+Seg"), ("Outputs_3", "+Depth")]

    for panel_ds, panel_name, panel_desc, use_labels_col in [("LiveUAMap", "Panel A", "LiveUAMap Dataset (Crowdsourced Coordinate Stream)", True),
                                                             ("ArcGIS", "Panel B", "ArcGIS Dataset (High-Precision Coordinate Stream)", False)]:

        ds_data = eval_df[eval_df['dataset'] == panel_ds]
        if ds_data.empty: continue

        # Calculate SAMGeo Baselines (minimum across Google and ESRI for the dataset)
        g_mae = ds_data['geosam_google_mae_total'].median()
        e_mae = ds_data['geosam_esri_mae_total'].median()
        g_mse = ds_data['geosam_google_mse_total'].median()
        e_mse = ds_data['geosam_esri_mse_total'].median()

        best_mae_base = min(g_mae, e_mae)
        best_mse_base = min(g_mse, e_mse)

        # Panel Header
        latex_out.extend([
            f"% ==================== PANEL {panel_name[-1]}: {panel_ds.upper()} ====================",
            r"\begin{tabular}{ll" + ("cccccccc" if use_labels_col else "cccccc") + "}",
            r"\toprule",
            f"\\multicolumn{{{10 if use_labels_col else 8}}}{{l}}{{\\textbf{{{panel_name}: {panel_desc}}}}} \\\\",
            r"\midrule"
        ])

        if use_labels_col:
            latex_out.extend([
                r"\textbf{Model Name} & \textbf{Config} & \multicolumn{4}{c}{\textbf{Mean Absolute Error (MAE) $\downarrow$}} & \multicolumn{4}{c}{\textbf{Mean Squared Error (MSE) $\downarrow$}} \\",
                r"\cmidrule(lr){3-6} \cmidrule(lr){7-10}",
                r" & & \textbf{Labels} & \textbf{No Lbls} & \textbf{+Seg} & \textbf{+Depth} & \textbf{Labels} & \textbf{No Lbls} & \textbf{+Seg} & \textbf{+Depth} \\",
                r"\midrule"
            ])
            active_configs = configs
        else:
            latex_out.extend([
                r"\textbf{Model Name} & \textbf{Config} & \multicolumn{3}{c}{\textbf{Mean Absolute Error (MAE) $\downarrow$}} & \multicolumn{3}{c}{\textbf{Mean Squared Error (MSE) $\downarrow$}} \\",
                r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
                r" & & \textbf{No Lbls} & \textbf{+Seg} & \textbf{+Depth} & \textbf{No Lbls} & \textbf{+Seg} & \textbf{+Depth} \\",
                r"\midrule"
            ])
            active_configs = [c for c in configs if c[0] != "Outputs_2"]

        # Model Rows
        models = ds_data['model_id'].unique()

        # Ensure targeted sort order
        model_order_keys = ["claude", "qwen", "gemma", "cosmos", "zhipu"]
        models_sorted = sorted(models, key=lambda x: next((i for i, k in enumerate(model_order_keys) if k in x.lower()), 99))

        for model in models_sorted:
            m_data = ds_data[ds_data['model_id'] == model]
            m_name, m_config = identify_model_meta(model)

            mae_vals_list = []
            mse_vals_list = []

            for cid, _ in active_configs:
                row = m_data[m_data['config_id'] == cid]
                mae_vals_list.append(row['llm_mae_total'].values[0] if not row.empty else np.nan)
                mse_vals_list.append(row['llm_mse_total'].values[0] if not row.empty else np.nan)

            mae_strs = [format_latex_val(v, mae_vals_list, best_mae_base) for v in mae_vals_list]
            mse_strs = [format_latex_val(v, mse_vals_list, best_mse_base) for v in mse_vals_list]

            row_str = f"{m_name:<18} & {m_config} & " + " & ".join(mae_strs) + " & " + " & ".join(mse_strs) + r" \\[0.5ex]"
            latex_out.append(row_str)

        # Baselines
        latex_out.extend([
            r"\midrule",
            f"SAMGeo Google Reference & 0.85B & \\multicolumn{{{4 if use_labels_col else 3}}}{{c}}{{\\small MAE: {g_mae:.2f}}} & \\multicolumn{{{4 if use_labels_col else 3}}}{{c}}{{\\small MSE: {g_mse:.2f}}} \\\\",
            f"SAMGeo ESRI Reference   & 0.85B & \\multicolumn{{{4 if use_labels_col else 3}}}{{c}}{{\\small MAE: {e_mae:.2f}}} & \\multicolumn{{{4 if use_labels_col else 3}}}{{c}}{{\\small MSE: {e_mse:.2f}}} \\\\",
            r"\bottomrule",
            r"\end{tabular}",
            ""
        ])

        if use_labels_col: latex_out.append(r"\vspace{1.2em}" + "\n")

    latex_out.append(r"\end{table*}")

    print("\n".join(latex_out))


🚀 INITIALIZING EVALUATION FOR DATASET: ARCGIS
📥 Loading ground truth...
   -> Ground truth mapped successfully: 892 features
📥 Loading and parsing Dual GeoSAM (Google & Esri) predictions...
   -> GeoSAM baseline vectors successfully loaded: 890 entities

📁 Sifting Pipeline Branch -> [Outputs_3] (llm+Seg+Depth)
   📂 Found 3 individual LLM variant model outputs
       🔹 Model: nuextract_parsed_Qwen_Qwe... | Shared N: 890 | LLM RMSE: 4.89 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_google_g... | Shared N: 890 | LLM RMSE: 3.78 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_sakamaki... | Shared N: 890 | LLM RMSE: 4.28 | Google RMSE: 4.39

📁 Sifting Pipeline Branch -> [Outputs_4] (llm+seg)
   📂 Found 3 individual LLM variant model outputs
       🔹 Model: nuextract_parsed_Qwen_Qwe... | Shared N: 890 | LLM RMSE: 5.41 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_google_g... | Shared N: 890 | LLM RMSE: 4.62 | Google RMSE: 4.39
       🔹 Model: nuextract_parsed_sakamaki... 